# Data Explpoite

In [1]:
import sys
import os
from pathlib import Path
# Remonte d'un niveau pour atteindre la racine (parent de /notebook)
sys.path.insert(0, os.path.abspath(".."))

In [2]:
from src.data.schema import TRANSACTIONS_SCHEMA
from pyspark.sql import SparkSession
from src.config import SparkConfig

cfg = SparkConfig()
spark = (
        SparkSession.builder
            .appName(cfg.app_name)
            .master(cfg.master)
            .config("spark.sql.shuffle.partitions", str(cfg.shuffle_partitions))
            .config("spark.sql.adaptive.enabled", str(cfg.adaptive_enabled).lower())
            .config("spark.sql.adaptive.coalescePartitions.enabled",
                    str(cfg.adaptive_enabled).lower())
            .config("spark.driver.memory", cfg.driver_memory)
            .config("spark.executor.memory", cfg.executor_memory)
            .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

In [3]:
path = Path().resolve().parent / "data" / "raw" / "online_retail_II.csv"

df = (
    spark.read
         .schema(TRANSACTIONS_SCHEMA)
         .option("header", True)
         .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
         .option("nullValue", "")
         .option("mode", "DROPMALFORMED")
         .csv(str(path))
)

In [4]:
df.show()

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+
|   489434|    85048|15CM CHRISTMAS GL...|      12|2009-12-01 07:45:00|     6.95|     13085|United Kingdom|
|   489434|   79323P|  PINK CHERRY LIGHTS|      12|2009-12-01 07:45:00|     6.75|     13085|United Kingdom|
|   489434|   79323W| WHITE CHERRY LIGHTS|      12|2009-12-01 07:45:00|     6.75|     13085|United Kingdom|
|   489434|    22041|"RECORD FRAME 7""...|      48|2009-12-01 07:45:00|      2.1|     13085|United Kingdom|
|   489434|    21232|STRAWBERRY CERAMI...|      24|2009-12-01 07:45:00|     1.25|     13085|United Kingdom|
|   489434|    22064|PINK DOUGHNUT TRI...|      24|2009-12-01 07:45:00|     1.65|     13085|United Kingdom|
|   489434|    21871| SAVE T

In [5]:
from pyspark.sql import DataFrame, functions as F

df = (
    df.filter(F.col('CustomerID').isNotNull())
      .filter(F.col("Quantity") > 0)
      .filter(F.col("UnitPrice") > 0)
      .filter(~F.col("InvoiceNo").startswith("C"))
      .withColumn("Revenue", F.col("Quantity") * F.col("UnitPrice"))
      .dropDuplicates()
    )

In [6]:
total = df.count()
nulls = (
        df.select([
            F.sum(F.col(c).isNull().cast("int")).alias(c)
            for c in df.columns
        ]).collect()[0].asDict()
    )

In [7]:
nulls

{'InvoiceNo': 0,
 'StockCode': 0,
 'Description': 0,
 'Quantity': 0,
 'InvoiceDate': 0,
 'UnitPrice': 0,
 'CustomerID': 0,
 'Country': 0,
 'Revenue': 0}

In [8]:
df.printSchema()

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Revenue: double (nullable = true)



In [9]:
n_rows = df.count()
n_cust = df.select("CustomerID").distinct().count()
n_inv  = df.select("InvoiceNo").distinct().count()
n_sku  = df.select("StockCode").distinct().count()

In [10]:
print(f"Lignes        : {n_rows:>12,}")
print(f"Clients       : {n_cust:>12,}")
print(f"Factures      : {n_inv:>12,}")
print(f"Produits      : {n_sku:>12,}")

Lignes        :      779,425
Clients       :        5,878
Factures      :       36,969
Produits      :        4,631


In [11]:
df.agg(
    F.min("InvoiceDate").alias("date_min"),
    F.max("InvoiceDate").alias("date_max"),
).show(truncate=False)


+-------------------+-------------------+
|date_min           |date_max           |
+-------------------+-------------------+
|2009-12-01 07:45:00|2011-12-09 12:50:00|
+-------------------+-------------------+



In [12]:
(df.groupBy("Country")
    .agg(F.round(F.sum("Revenue"), 2).alias("revenu_total"),
        F.countDistinct("CustomerID").alias("nb_clients"))
    .orderBy(F.col("revenu_total").desc())
    .show(5, truncate=False))

+--------------+-------------+----------+
|Country       |revenu_total |nb_clients|
+--------------+-------------+----------+
|United Kingdom|1.438923492E7|5350      |
|EIRE          |616570.54    |5         |
|Netherlands   |554038.09    |22        |
|Germany       |425019.71    |107       |
|France        |348768.96    |95        |
+--------------+-------------+----------+
only showing top 5 rows


In [13]:
(df.groupBy("StockCode", "Description")
    .agg(F.sum("Quantity").alias("qte_totale"),
        F.round(F.sum("Revenue"), 2).alias("revenu_total"))
    .orderBy(F.col("qte_totale").desc())
    .show(5, truncate=False))

df.select("Quantity", "UnitPrice", "Revenue").summary(
    "count", "mean", "stddev", "min", "25%", "50%", "75%", "max"
).show()


+---------+----------------------------------+----------+------------+
|StockCode|Description                       |qte_totale|revenu_total|
+---------+----------------------------------+----------+------------+
|84077    |WORLD WAR 2 GLIDERS ASSTD DESIGNS |105185    |24098.03    |
|85123A   |WHITE HANGING HEART T-LIGHT HOLDER|91757     |247048.01   |
|23843    |PAPER CRAFT , LITTLE BIRDIE       |80995     |168469.6    |
|84879    |ASSORTED COLOUR BIRD ORNAMENT     |78234     |124351.86   |
|23166    |MEDIUM CERAMIC TOP STORAGE JAR    |77916     |81416.73    |
+---------+----------------------------------+----------+------------+
only showing top 5 rows
+-------+------------------+------------------+------------------+
|summary|          Quantity|         UnitPrice|           Revenue|
+-------+------------------+------------------+------------------+
|  count|            779425|            779425|            779425|
|   mean|13.489369727683869|3.2184879853755906|22.291823161943626|
| 

In [14]:
from datetime import datetime, timedelta
from typing import Optional

from pyspark.sql import DataFrame, functions as F
from src.config import FeaturesConfig

cfg = FeaturesConfig
cutoff = datetime.strptime(cfg.cutoff_date, "%Y-%m-%d")

In [15]:
cfg.cutoff_date

'2010-10-01'

In [16]:
cutoff = F.lit(cutoff)
horizon_end = cutoff + timedelta(days=cfg.horizon_days)
past = df.filter(F.col("InvoiceDate") < cutoff)
future = df.filter(
    (F.col("InvoiceDate") >= cutoff) &
    (F.col("InvoiceDate") < F.lit(horizon_end))
)

In [17]:
cutoff

Column<'2010-10-01 00:00:00.0'>

In [18]:
df.filter(F.col("InvoiceDate") < cutoff).show()

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+------------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|           Revenue|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+------------------+
|   489434|    21232|STRAWBERRY CERAMI...|      24|2009-12-01 07:45:00|     1.25|     13085|United Kingdom|              30.0|
|   489434|    22064|PINK DOUGHNUT TRI...|      24|2009-12-01 07:45:00|     1.65|     13085|United Kingdom|39.599999999999994|
|   489435|    22353|LUNCHBOX WITH CUT...|      12|2009-12-01 07:46:00|     2.55|     13085|United Kingdom|30.599999999999998|
|   489437|    37370|RETRO COFFEE MUGS...|      12|2009-12-01 09:08:00|     1.25|     15362|United Kingdom|              15.0|
|   489437|    10002|INFLATABLE POLITI...|      12|2009-12-01 09:08:00|     0.85|     15362|United Kingdom|    

In [23]:
future.show()

+---------+---------+--------------------+--------+-------------------+---------+----------+---------------+------------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|        Country|           Revenue|
+---------+---------+--------------------+--------+-------------------+---------+----------+---------------+------------------+
|   524862|    22553|PLASTERS IN TIN S...|      12|2010-10-01 08:39:00|     1.65|     12417|          Spain|19.799999999999997|
|   524863|    22556|PLASTERS IN TIN C...|      12|2010-10-01 08:57:00|     1.65|     12817|        Austria|19.799999999999997|
|   524864|    22117|METAL SIGN HER DI...|       6|2010-10-01 09:02:00|     2.95|     14564|Channel Islands|17.700000000000003|
|   524864|    22551|PLASTERS IN TIN S...|      12|2010-10-01 09:02:00|     1.65|     14564|Channel Islands|19.799999999999997|
|   524864|    22665|RECIPE BOX BLUE S...|       6|2010-10-01 09:02:00|     2.95|     14564|Channel Isla

In [20]:
features = (
    past.groupBy("CustomerID")
        .agg(
            F.datediff(cutoff, F.max("InvoiceDate")).alias("recency"),
            F.countDistinct("InvoiceNo").alias("frequency"),
            F.round(F.sum("Revenue"), 2).alias("monetary"),
            F.round(F.avg("Revenue"), 2).alias("avg_basket"),
            F.countDistinct("StockCode").alias("n_products"),
            F.countDistinct("Country").alias("n_countries"),
            F.datediff(cutoff, F.min("InvoiceDate")).alias("tenure_days"),
        )
)

In [21]:
active_in_future = (
    future.select("CustomerID")
            .distinct()
            .withColumn("active_future", F.lit(1))
)

In [22]:
features.show()
active_in_future.show()

+----------+-------+---------+--------+----------+----------+-----------+-----------+
|CustomerID|recency|frequency|monetary|avg_basket|n_products|n_countries|tenure_days|
+----------+-------+---------+--------+----------+----------+-----------+-----------+
|     14110|      1|       21|  7304.8|     29.94|       126|          1|        304|
|     17984|     61|        2|  437.83|      4.09|        95|          1|        304|
|     13442|    233|        3|  419.87|     11.05|        33|          1|        304|
|     17428|     28|        9| 9074.21|     37.65|       180|          1|        304|
|     15485|    247|        3| 1222.92|     34.94|        29|          1|        304|
|     13269|      5|       11| 2497.13|     17.84|        96|          1|        303|
|     15680|     38|        6| 2011.85|      15.6|       101|          1|        303|
|     14204|    102|        2|  711.11|      5.12|       131|          1|        303|
|     17841|      3|       72|22058.24|      5.65|    